# QC: Conditional Diffusion Model Comparison

Compare two training scenarios for conditional facies generation from RMS amplitude:

| Scenario | `predict_type` | Run ID | Description |
|---|---|---|---|
| **A** | `epsilon` | `20260507_141532` | Model predicts noise (ε), subtract to get image |
| **B** | `x_start` | `20260510_112719` | Model predicts output image (x₀) directly |

Both models share the same UNet architecture, beta schedule, and training hyperparameters.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../scripts'))

%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 150

from flumy_check_results import ResultsQC

## 1. Initialize QC Object

In [ ]:
qc = ResultsQC(
    run_epsilon="20260507_141532",
    run_xstart="20260510_112719",
    base_dir="/mnt/sda_data/tharitt/diffsim",
)
print(f"Test dataset: {len(qc.test_dataset)} samples")
print(f"Image size: {qc.image_size}x{qc.image_size}")
print(f"Device: {qc.device}")

## 2. Training Loss Curves
Read TensorBoard event files and plot smoothed training/validation loss.

In [ ]:
fig = qc.plot_losses(
    smoothing=0.95,
    figsize=(14, 5),
    # save_path="../assets/loss_comparison.png",
)

## 3. Input Data Overview
Show ground truth facies maps and their RMS conditioning inputs.

In [ ]:
fig = qc.plot_input_data(
    index=200,
    num_samples=4,
    figsize=(18, 5),
    # save_path="../assets/input_data_overview.png",
)

## 4. Single-Sample Inference Comparison
Run both models on a test sample and compare side-by-side with the ground truth.

In [ ]:
fig = qc.infer_and_compare_with_rms(
    index=202,
    use_ddim=True,
    ddim_steps=50,
    seed=42,
    figsize=(20, 4),
    # save_path="../assets/comparison_sample_0.png",
)

In [ ]:
# Try another sample
fig = qc.infer_and_compare_with_rms(
    index=5,
    seed=42,
    # save_path="../assets/comparison_sample_5.png",
)

## 5. Batch Comparison Grid
Multiple samples in a single figure (good for slides).

In [ ]:
fig = qc.batch_compare(
    indices=[0, 3, 7, 12, 18],
    use_ddim=True,
    ddim_steps=50,
    seed=42,
    # save_path="../assets/batch_comparison.png",
)

## 6. Facies Proportions
Compare facies distribution (mud/bank/sand %) between GT and both models.

In [ ]:
fig = qc.plot_facies_distribution(
    index=204,
    seed=42,
    # save_path="../assets/facies_distribution.png",
)

## 7. Inference from an Arbitrary RMS File
Load a standalone `.npy` RMS map and generate facies from both models.

In [ ]:
# Point to any RMS .npy file
rms_file = "../data/flumy_dataset/test/rms/" + os.listdir("../data/flumy_dataset/test/rms/")[0]
print(f"Using RMS file: {rms_file}")

fig = qc.infer_from_rms_file(
    rms_path=rms_file,
    seed=42,
    # save_path="../assets/inference_external_rms.png",
)

## 8. Save All Figures for Slides
Uncomment and run to export all figures at once.

In [ ]:
# output_dir = "../assets/qc_figures"
# os.makedirs(output_dir, exist_ok=True)
#
# qc.plot_losses(save_path=f"{output_dir}/loss_comparison.png")
# qc.plot_input_data(index=0, num_samples=4, save_path=f"{output_dir}/input_data.png")
# qc.infer_and_compare_with_rms(index=0, seed=42, save_path=f"{output_dir}/compare_sample0.png")
# qc.batch_compare(indices=[0, 3, 7, 12, 18], seed=42, save_path=f"{output_dir}/batch_grid.png")
# qc.plot_facies_distribution(index=0, seed=42, save_path=f"{output_dir}/facies_dist.png")
# print(f"All figures saved to {output_dir}/")